## Hybrid Retrieval Tuning

In [ ]:
import os
from getpass import getpass
os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
%pip install .

In [ ]:
# Re-run after pushing code/config changes
!git pull
%pip install .

In [ ]:
# Download processed data (incl. pre-segmented corpus_pyvi.csv / eval_pyvi.csv)
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip

In [ ]:
!pip install -U huggingface_hub
!hf auth login --token ${HF_TOKEN}

## Run the sweep

In [ ]:
!cat configs/hybrid-tuning/hybrid_sweep.yaml

In [ ]:
# VRAM-aware batch sizes (prints report; tune_hybrid uses the same logic internally)
!python scripts/update_batch_size.py --vram auto

In [ ]:
!python scripts/tune_hybrid.py configs/hybrid-tuning/hybrid_sweep.yaml

## Results

In [ ]:
!cat results/hybrid-tuning/scores.json | python3 -m json.tool | tail -60

In [ ]:
# Persist scores + cache to Drive (survives Colab reset)
!mkdir -p /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning
!cp -r results/hybrid-tuning/* /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning/

In [ ]:
!cp results/hybrid-tuning/scores.json /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning/scores.json

In [ ]:
!mv results/hybrid-tuning/cache/* results/hybrid-tuning/

In [ ]:
!git config user.email "vhphatdz@gmail.com"
!git config user.name "phatv1"
!git add results/hybrid-tuning/scores.json
!git commit -m "results: hybrid retrieval tuning (bm25+dense+cross sweep)" || echo "nothing to commit"
!git push || echo "push skipped"